# 1. 프로젝트 개요: 수출 전략 보고서 자동 생성 시스템

이 시스템은 **사용자가 국가와 HS코드를 입력하면**, 해당하는 **해외 시장 보고서를 자동으로 생성**하는 프로젝트입니다.

보고서는 다음의 세 가지 데이터를 결합하여 만듭니다:

1) 과거 데이터 기반 자료  
- KATI(한국농수산식품유통공사) PDF  
- KOTRA(대한무역투자진흥공사) 해외시장조사보고서 PDF  
- 국가정보 JSON 데이터
- 규제 csv

이 데이터는 **Vector Database**(벡터 데이터베이스)에 저장되어 AI가 필요한 내용을 빠르게 찾아볼 수 있도록 구성됩니다.

2) 최신성 기반 자료  
웹에서 검색된 최신 규제/시장/트렌드 등의 정보를 GPT로 정리합니다.  
이를 **Deep Research Engine**이 담당합니다. 
최신 규제는 tavily api key를 이용하여 받아옴

3) 최종 보고서 작성  
    1) 국가정보 기반 초안 보고서 생성  
    2) VectorDB의 정량 데이터로 보강  
    3) Deep Research로 최신 데이터 통합  
    4) KOTRA 스타일의 최종 보고서 조립  

- 전체 구조 흐름도

In [ ]:
# 전체 구조 흐름도
"""
사용자 입력
│
▼
[1] 데이터 로딩(DataLoader)
│ PDF→텍스트, JSON 로드, chunk 분할
│
▼
[2] 벡터DB(VectorDB)
│ 문서를 embedding 후 저장/로드
│
▼
[3] 통합 검색 엔진(Integrated Search)
│ ├ QueryGenerator (검색 쿼리 생성 GPT)
│ ├ VectorDB 검색
│ └ 표·이미지 힌트 수집
│
▼
[4] Deep Research Engine
│ 웹검색 → 최신 규제/동향 구조화
│
▼
[5] Report Generator
├ 초안 생성 (GPT)
├ RAG 기반 보강 (KATI/KOTRA)
├ Deep Research 통합
└ 최종 보고서 조립
"""

In [ ]:
"""
📁 데이터 준비
data_loader.py → PDF/JSON 로드 및 chunk 생성

📁 벡터DB 구축 (1회)
build_vectordb.py → vectordb_manager.py로 저장

📁 사용자 입력
integrated_search.py 실행
↓
QueryGenerator로 쿼리 생성
↓
VectorDB 섹션별 검색 (search_engine.py)
↓
표·이미지 힌트 수집
↓
국가정보 JSON 로드
↓
result 패키지 생성

📁 Deep Research
deep_research.py → 최신 규제/가격/리스크 분석

📁 보고서 생성
report_generator.py
1) 초안
2) RAG 보강
3) 최신 업데이트
4) 최종 조립

📁 최종 출력
output/final_report.txt 생성
"""

| 파일명                      | 역할                   | 실행 시점                |
| ------------------------ | -------------------- | -------------------- |
| **data_loader.py**       | PDF/JSON 로드 및 텍스트 청킹 | VectorDB 구축 전/검색 시   |
| **vectordb_manager.py**  | VectorDB 저장·로드·검색    | 전체                   |
| **build_vectordb.py**    | 벡터DB 구축 (1회 실행)      | 프로젝트 초반              |
| **search_engine.py**     | VectorDB 검색 엔진       | integrated_search 내부 |
| **integrated_search.py** | 사용자 입력 → 전체 검색 통합    | 보고서 생성 1단계           |
| **deep_research.py**     | 웹 최신 정보 수집           | 보고서 생성 2단계           |
| **report_generator.py**  | 보고서 초안→보강→최신→최종      | 보고서 생성 3~4단계         |
| **report_pipeline.py**   | 전체 파이프라인 실행          | 최종 실행                |
| **mapping.py**           | 국가·HS코드 등 매핑         | 유틸                   |


# 2. 사용자 입력 예시

이 시스템의 입력값은 단 4가지입니다.

```json
{
  "country": "일본",  # 드롭다운, 단일 선택
  "hs_code": "2008190000", # 10단위 입력 
  "extra_analysis": ["시장 리스크", "가격 추세"], # 분석 추가 선택, 중복 선택 가능 
  "sns_keyword": "바나나" # sns 트렌드 시각화 용 
}


# 3. 데이터 로딩 과정 설명 (DataLoader)

- 데이터 로딩(DataLoader) 이해하기
    - DataLoader는 **PDF와 JSON 파일을 불러오고, 텍스트를 Chunk 단위로 자르는 역할**을 합니다.
    - 이 과정에서는 AI 프롬프트가 사용되지 않습니다.

-  DataLoader가 하는 일 
    1) PDF 파일을 읽는다  
    2) 표지 / 목차 / 저작권 페이지 등 필요 없는 페이지 제거  
    3) 본문 텍스트를 청크(chunk)라는 작은 단위로 자른다  
    4) 각 청크마다 metadata(국가명, 출처, PDF파일명, 페이지 범위)를 붙인다  
    5) VectorDB에 저장할 수 있는 구조로 반환한다  


# 4. 벡터데이터베이스(VectorDB)란?

VectorDB는 텍스트 정보를 **벡터(숫자의 배열)**로 변환해 저장하고,  
AI가 원하는 내용을 빠르게 찾아주는 검색 시스템입니다.


- VectorDB가 하는 일
  1) 생성
    - Chunk 텍스트 + metadata → embedding(768차원 숫자 배열)
    - Chroma / Pinecone 등에 저장

  2) 로드
    - 이미 저장된 VectorDB를 메모리에 다시 불러오기

  3) 검색
    - "일본 시장 HS 2008190000 시장규모..." 같은 쿼리를 넣으면 가장 유사한 5개의 문서를 찾아 반환한다.

- VectorDB는 AI와 다릅니다!
  - VectorDB는 **기억장치(하드디스크)** 역할
  - GPT는 **두뇌(생각을 정리)** 역할

두 개가 함께 작동해야 RAG가 됩니다. 


# 5. QueryGenerator: 검색 최적화 AI 프롬프트

VectorDB는 키워드 기반 검색이기 때문에,  
사용자 입력을 **AI가 '최적화된 검색 문장'으로 바꾸어주는 과정**이 필요합니다.

- 역할
    입력 예시:

    ```python
    {
    "country": "일본",
    "hs_code": "2008190000",
    "extra_analysis": ["시장 리스크", "가격 추세"]
    }


# 6. Integrated Search 엔진 설명

이 단계에서는 다음 3가지 작업이 동시에 일어납니다.


1) QueryGenerator로 검색 문장 생성
    - VectorDB가 이해할 수 있는 한국어 검색 문장 생성

2) VectorDB에서 문서 검색
    - 시장 규모
    - 경쟁 현황
    - 규제 설명
    - 품목 정보 
    등이 필요한 만큼 검색됨

3) 표·이미지 힌트 수집
    PDF의 다음 정보도 함께 수집:
    - table_pages: 표가 있는 페이지
    - image_pages: 그래프가 있는 페이지
    - file_name: 데이터 출처

    이 정보는 **최종 보고서에서 수치의 근거를 명확히 적기 위해 필수**
    예:  (출처: 2023_kati_JP.pdf, 페이지 7-9 표)


# 7. Deep Research Engine: 최신성 정보를 담당

KATI/KOTRA 자료는 보통 최근 1~2년까지입니다.  
하지만 시장·규제는 **매일 업데이트됩니다.**

따라서 최신 정보를 자동으로 가져오는 단계가 필요합니다.

- Deep Research의 입력
  - 국가
  - 제품명
  - HS 코드

- Deep Research의 출력
  예:

  ```json
  {
    "latest_info": "2025년 3월부터 견과류 알레르기 표시 의무화 시행",
    "source": "일본 후생노동성",
    "confidence": "high",
    "date": "2025-03-01"
  }


# 8. Report Generator — 4단계 보고서 생성

보고서는 아래 단계로 생성됩니다.


- 1단계: 초안 생성(generate_initial_draft)
    - 국가정보 JSON을 기반으로 기본 틀만 만든다.

- 2단계: VectorDB 기반 보정(enhance_with_documents)
    - KATI/KOTRA 문서의 숫자, 표, 그래프를 활용해 구체화한다.

    예: 
    | 연도 | 시장규모 | 출처 |
    |------|----------|------|
    | 2023 | 6,580억 달러 | 2023_kati_JP.pdf p.7 |  

   
- 3단계: Deep Research 통합(integrate_deep_research)
    - 2025년 기준 최신 규제·가격·리스크 정보를 반영

- 4단계: 최종 조립(assemble_final_report)
    - KOTRA 해외시장조사보고서 스타일의 전문 보고서 완성


# 9. 전체 파이프라인 한눈에 보기



```python
result = search_engine.search_all(test_input)

dr = DeepResearchEngine()
dr_result = dr.run_all_research(...)

rg = ReportGenerator()

initial = rg.generate_initial_draft(...)
enhanced = rg.enhance_with_documents(initial, ...)
deep_added = rg.integrate_deep_research(enhanced, dr_result)
final_report = rg.assemble_final_report(deep_added, ...)


# 10. 최종 요약

이 프로젝트는 국제무역·식품수출 실무자들이  
**국가별 시장 특성 + 최신 규제 + 품목별 시장 수치**를  
단 10초 만에 생성할 수 있게 해주는 자동 보고서 생성 시스템입니다.


- 이 시스템의 강점

    1. 데이터 + AI 융합
        - VectorDB(RAG) → 과거 문서의 수치 기반 근거
        - Deep Research → 최신 정부기관 규제
        - GPT → 전문가 스타일 보고서 조립

    2. 환각 방지 구조
        - RAG 기반 정보만 사용하도록 강제
        - 출처·페이지 기록 기능
        - Deep Research는 “신뢰도”까지 기록

    3. KOTRA 보고서 품질 재현
        - 객관적 데이터 중심
        - 시장 분석 → 구조 분석 → 전략 제안 흐름




In [ ]:
"""
pip install python-dotenv
pip install langchain
pip install langchain-openai
pip install chromadb
pip install pymupdf
pip install tavily-python
"""

In [ ]:
prompt = f"""
You are a professional analyst specializing in writing export strategy reports.

You will be given **two structured inputs**:
1) Official country information data (JSON)
2) User request information (JSON)

These are the ONLY data sources you are allowed to use.
You MUST strictly follow all constraints below.

----------------------------------------
【 COUNTRY INFORMATION JSON 】
{json.dumps(country_info, ensure_ascii=False, indent=2)}

【 USER REQUEST INFORMATION 】
{json.dumps(request_info, ensure_ascii=False, indent=2)}
----------------------------------------

Your task:
Generate an **initial draft report** that includes ONLY the following sections:

1. Country Overview  
2. Food Market Size & Growth Trend  
3. Import Structure & Trade with Korea  
4. Consumer Behavior / Key Food Culture Patterns  
5. Basic Information on FTA, Tariffs, and Import Regulations  
6. Market Fit Assessment related to the target HS code  

----------------------------------------
🚨 STRICT ANTI-HALLUCINATION RULES (DO NOT BREAK)  
----------------------------------------

1. **Use ONLY information explicitly provided in the JSON inputs.**
2. **DO NOT invent, infer, assume, estimate, or fabricate any facts, numbers, trends, timelines, or examples.**
3. **If a required detail does not exist in the JSON, clearly write:  
   “(No data provided in JSON)”**
4. **Do not use external knowledge**, even if you believe it is true.
5. **Do not generalize based on stereotypes, common knowledge, or assumptions.**
6. **Do not rewrite or reinterpret missing values. State them as missing.**
7. **Do not add additional sections, disclaimers, or formatting not requested.**

----------------------------------------
Additional instructions:
- Maintain a factual, analytical writing style.
- Structure the draft clearly according to the six required headings.
- DO NOT include data from KATI, KOTRA, or any Deep Research sources at this stage.
- Your output must be based **strictly on the JSON content provided above**.

Produce the final draft now.
"""


In [ ]:
""" 
당신은 국가별 수출 전략 보고서를 작성하는 전문 분석가입니다.

아래 두 가지 JSON 데이터가 제공됩니다:
1) 공식 국가정보 데이터(JSON)
2) 사용자 요청 정보(JSON)

이 두 데이터는 **당신이 사용할 수 있는 유일한 정보**입니다.
아래 제약을 반드시 지키십시오.

----------------------------------------
【 국가정보 JSON 】
{json.dumps(country_info, ensure_ascii=False, indent=2)}

【 사용자 요청 정보 JSON 】
{json.dumps(request_info, ensure_ascii=False, indent=2)}
----------------------------------------

요구 작업:
아래 6개 항목만을 포함하는 **초안 보고서(initial draft)** 를 작성하십시오.

1. 국가 개요
2. 식품 시장 규모 및 성장 흐름
3. 수입 구조 및 한국과의 교역 현황
4. 소비자 성향 및 식문화 핵심 특징
5. FTA·관세·수입 규제 관련 기본 정보
6. 분석 대상 품목(HS 코드)의 시장 적합성 평가

----------------------------------------
🚨 절대 위반 금지: 강력한 할루시네이션 방지 규칙  
----------------------------------------

1. 제공된 JSON에 **명시적으로 존재하는 정보만** 사용하십시오.
2. **추론, 가정, 외삽, 일반 상식 기반의 판단을 절대 하지 마십시오.**
3. **숫자, 통계, 연도, 정책, 규제 등은 JSON에 없으면 절대 만들지 마십시오.**
4. JSON에 없는 내용은 반드시 다음과 같이 명시하십시오:
   - “(JSON에 해당 정보 없음)”
5. 외부 지식(인터넷, 학습 지식, 경험 등)을 절대 활용하지 마십시오.
6. 필요한 데이터가 누락된 경우 임의로 보완하거나 “있었다고 가정”하지 마십시오.
7. 요청된 6개 항목 외에 추가적인 목차나 설명을 작성하지 마십시오.

----------------------------------------
추가 지침:
- 서술은 분석적이고 객관적인 톤으로 작성하십시오.
- 반드시 지정된 6개 섹션 구조를 따르십시오.
- 이 단계에서는 KATI·KOTRA·Deep Research 정보는 절대 반영하지 마십시오.
- 출력은 하나의 초안 보고서 문서여야 합니다.

초안 보고서를 작성하십시오.
"""



In [1]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from datetime import date

# ---- 1. 보고서 메타 정보 (여기만 바꿔 쓰면 됨) ----
project_title = "일본 견과류 수출 전략 보고서"
sub_title = "HS 2008190000 기준 / GlobalPath AI 자동 생성"
client_name = "귀사명 또는 과제명"
author_name = "작성자: 홍길동"
today = date.today().strftime("%Y-%m-%d")

# ---- 2. 프레젠테이션 생성 ----
prs = Presentation()
slide_layout = prs.slide_layouts[6]  # Blank
slide = prs.slides.add_slide(slide_layout)

# 배경 색 설정 (연한 파랑)
bg_fill = slide.background.fill
bg_fill.solid()
bg_fill.fore_color.rgb = RGBColor(240, 245, 252)

# ---- 3. 타이틀 텍스트 박스 ----
left = Inches(1.0)
top = Inches(2.0)
width = Inches(8.0)
height = Inches(2.0)

title_box = slide.shapes.add_textbox(left, top, width, height)
title_tf = title_box.text_frame
title_tf.text = project_title

title_run = title_tf.paragraphs[0].runs[0]
title_run.font.size = Pt(40)
title_run.font.bold = True
title_run.font.color.rgb = RGBColor(20, 40, 80)

# ---- 4. 서브타이틀 ----
subtop = Inches(3.4)
sub_box = slide.shapes.add_textbox(left, subtop, width, Inches(1.0))
sub_tf = sub_box.text_frame
sub_tf.text = sub_title

sub_run = sub_tf.paragraphs[0].runs[0]
sub_run.font.size = Pt(20)
sub_run.font.color.rgb = RGBColor(60, 80, 120)

# ---- 5. 하단 정보 (작성자, 날짜, 기관명 등) ----
bottom = Inches(6.5)
info_box = slide.shapes.add_textbox(left, bottom, width, Inches(1.0))
info_tf = info_box.text_frame

for line in [client_name, author_name, f"작성일: {today}"]:
    p = info_tf.add_paragraph()
    p.text = line
    p.font.size = Pt(14)
    p.font.color.rgb = RGBColor(80, 80, 80)

# 첫 줄은 공백이라면 제거
if info_tf.paragraphs and not info_tf.paragraphs[0].text:
    info_tf._element.remove(info_tf.paragraphs[0]._p)

# ---- 6. 저장 ----
prs.save("report_cover.pptx")
print("✅ report_cover.pptx 생성 완료")


✅ report_cover.pptx 생성 완료


In [ ]:
# !pip install reportlab

In [4]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.colors import HexColor

def create_report_cover(output_path="report_cover.pdf", 
                        title="ANNUAL REPORT", 
                        year="2023", 
                        subtitle="Lorem Ipsum", 
                        description="Lorem Ipsum is standard text placed to demonstrate graphical or visual presentation elements."):
    
    # A4 사이즈
    width, height = A4  

    # 캔버스 생성
    c = canvas.Canvas(output_path, pagesize=A4)

    # ===== 배경 색 =====
    c.setFillColor(HexColor("#E8F5F9"))   # 연한 민트색
    c.rect(0, 0, width, height, fill=1, stroke=0)

    # ===== 상단 큰 파란 박스 =====
    c.setFillColor(HexColor("#7DA7FF"))
    c.rect(40, height - 160, 60, 110, fill=1, stroke=0)

    # ===== 오른쪽 세로 파란 라인 =====
    c.setFillColor(HexColor("#7DA7FF"))
    c.rect(width - 80, 200, 18, height - 360, fill=1, stroke=0)

    # ===== 제목 텍스트 =====
    c.setFillColor(HexColor("#43A4AE"))
    c.setFont("Helvetica-Bold", 32)
    c.drawString(80, height - 240, title)

    c.setFillColor(HexColor("#5B9BD5"))
    c.setFont("Helvetica-Bold", 36)
    c.drawString(80, height - 280, year)

    # ===== 중간 장식 원 세 개 =====
    c.setLineWidth(3)
    c.setStrokeColor(HexColor("#7DA7FF"))
    circle_y = height - 350
    c.circle(150, circle_y, 18)
    c.circle(230, circle_y, 18)
    c.circle(310, circle_y, 18)

    # ===== 하단 텍스트 =====
    c.setFillColor(HexColor("#3E73B9"))
    c.setFont("Helvetica-Bold", 16)
    c.drawString(80, height - 430, subtitle)

    c.setFont("Helvetica", 10)
    text_x = 80
    text_y = height - 450
    for line in description.split("\n"):
        c.drawString(text_x, text_y, line)
        text_y -= 14

    # ===== 하단 파란 장식 막대 =====
    c.setFillColor(HexColor("#7DA7FF"))
    c.rect(80, 70, width - 160, 18, fill=1, stroke=0)

    # PDF 저장
    c.save()
    print(f"PDF 표지 생성 완료: {output_path}")


# 실행
create_report_cover()


PDF 표지 생성 완료: report_cover.pdf
